# Results Analysis
### GP Uncertainty for Novel Defect Detection — ST7085CEM Task 1

| Tag | Destination |
|---|---|
| `[REPORT]`   | Paper body |
| `[APPENDIX]` | Supplementary appendix |
| `[CODE ONLY]`| Not cited in paper |

**Run order:** This notebook requires `python run_pipeline.py` to have completed (phases 0–6).


## 0. Setup

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import roc_curve, auc

warnings.filterwarnings("ignore")

ROOT     = Path("..")
EVAL_DIR = ROOT / "results" / "evaluation"
FIG_DIR  = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
})

# ── load GP fold results ──
fold_df  = pd.read_csv(EVAL_DIR / "fold_results.csv")
cat_df   = pd.read_csv(EVAL_DIR / "category_summary.csv")
with open(EVAL_DIR / "evaluation_summary.json") as f:
    summary = json.load(f)

# ── load baseline fold results ──
baseline_dfs = {}
for name in ("ensemble", "mc_dropout"):
    p = EVAL_DIR / f"baseline_{name}_fold_results.csv"
    if p.exists():
        baseline_dfs[name] = pd.read_csv(p)

print(f"GP folds loaded:      {len(fold_df)}")
print(f"Baselines loaded:     {list(baseline_dfs.keys())}")
print(f"Overall AUROC (GP):   {summary['mean_auroc_do']:.3f} ± {summary['std_auroc_do']:.3f}")
for bname, bauc in summary.get("baseline_mean_auroc", {}).items():
    print(f"  {bname}: {bauc:.3f}")


## F1  [REPORT §6] — GP vs Baselines: Per-category AUROC

**Caption:** Mean AUROC (defect-only labelling) per category for the GP model
and each baseline. Error bars = ±1 SD over LODTO folds.
Dashed line = chance (0.5). GP uses posterior variance as the novelty score;
baselines use reconstruction error from autoencoders trained on normal images only.


In [ ]:
cats = sorted(fold_df["category"].unique())
x    = np.arange(len(cats))

METHOD_STYLES = {
    "GP (ours)":   {"color": "#1565C0", "label": "GP posterior variance (ours)"},
    "ensemble":    {"color": "#E65100", "label": "Deep ensemble (recon. error)"},
    "mc_dropout":  {"color": "#558B2F", "label": "MC dropout (recon. error)"},
}
n_methods = 1 + len(baseline_dfs)
width = 0.8 / n_methods

fig, ax = plt.subplots(figsize=(12, 4.5))

# GP bars
gp_means = [fold_df[fold_df.category == c]["auroc_do"].mean() for c in cats]
gp_stds  = [fold_df[fold_df.category == c]["auroc_do"].std()  for c in cats]
offset = -(n_methods - 1) / 2 * width
ax.bar(x + offset, gp_means, width, yerr=gp_stds, capsize=2.5,
       label="GP posterior variance (ours)", color="#1565C0",
       error_kw={"lw": 1.2}, alpha=0.9)

# Baseline bars
for k, (bname, bdf) in enumerate(baseline_dfs.items()):
    merged = bdf.groupby("category")["auroc_do"]
    b_means = [merged.mean().get(c, np.nan) for c in cats]
    b_stds  = [merged.std().get(c, np.nan)  for c in cats]
    off = offset + (k + 1) * width
    style = METHOD_STYLES.get(bname, {"color": "#607D8B"})
    label = {"ensemble": "Deep ensemble", "mc_dropout": "MC dropout"}.get(bname, bname)
    ax.bar(x + off, b_means, width, yerr=b_stds, capsize=2.5,
           label=label, color=style["color"],
           error_kw={"lw": 1.2}, alpha=0.75)

ax.axhline(0.5, ls="--", color="gray", lw=1, label="Chance")
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=38, ha="right", fontsize=9)
ax.set_ylabel("AUROC (defect-only)")
ax.set_title("Novel Defect Detection: GP vs Baselines — Per-category AUROC")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, 1.12)

plt.tight_layout()
plt.savefig(FIG_DIR / "F1_gp_vs_baselines_auroc.png")
plt.show()
print("Saved F1_gp_vs_baselines_auroc.png")


## F2  [REPORT §6] — AUROC Heatmap (Category × Held-out Defect Type)

**Caption:** AUROC (defect-only) for each LODTO fold.
Rows = MVTec AD categories; columns = held-out defect types.
Warm colours (→ red) indicate the GP readily distinguishes the novel type
from known defects in that category.


In [ ]:
pivot = fold_df.pivot_table(index="category", columns="held_out_type", values="auroc_do")

fig, ax = plt.subplots(figsize=(16, 6))
im = sns.heatmap(
    pivot, ax=ax, cmap="RdYlGn", center=0.5, vmin=0, vmax=1,
    annot=True, fmt=".2f", linewidths=0.3,
    cbar_kws={"label": "AUROC (defect-only)", "shrink": 0.8},
)
ax.set_xlabel("Held-out defect type (novel probe)", fontsize=10)
ax.set_ylabel("Category", fontsize=10)
ax.set_title("Leave-One-Defect-Type-Out AUROC — GP Posterior Variance", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "F2_auroc_heatmap.png")
plt.show()
print("Saved F2_auroc_heatmap.png")


## F3  [REPORT §6] — Aggregate ROC Curves

**Caption:** Macro-averaged ROC curves over 72 LODTO folds for each method.
Shaded band = ±1 SD across folds. The Area Under the Curve (AUROC) for each
method is shown in the legend. Defect-only labelling: novel defect type = positive,
known defect types = negative.


In [ ]:
def fold_roc(novel_scores, known_scores):
    scores = np.concatenate([novel_scores, known_scores])
    labels = np.concatenate([np.ones(len(novel_scores)), np.zeros(len(known_scores))])
    if len(np.unique(labels)) < 2:
        return None, None, 0.0
    fpr, tpr, _ = roc_curve(labels, scores)
    return fpr, tpr, auc(fpr, tpr)

def interp_roc(fpr_list, tpr_list, n=200):
    mean_fpr = np.linspace(0, 1, n)
    tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fpr_list, tpr_list)]
    return mean_fpr, np.array(tprs)

fig, ax = plt.subplots(figsize=(6, 5.5))

METHOD_COLORS = {
    "GP (ours)":  "#1565C0",
    "ensemble":   "#E65100",
    "mc_dropout": "#558B2F",
}
METHOD_LABELS = {
    "GP (ours)":  "GP posterior variance (ours)",
    "ensemble":   "Deep ensemble",
    "mc_dropout": "MC dropout",
}

# ── GP ROC ──
gp_fprs, gp_tprs = [], []
with open(EVAL_DIR / "gp_pca16.json") as f:
    gp_raw = json.load(f)

for r in gp_raw:
    novel = np.array(r["test_scores"])
    known = np.array(r.get("known_scores", []))
    if len(known) == 0:
        continue
    fpr, tpr, _ = fold_roc(novel, known)
    if fpr is not None:
        gp_fprs.append(fpr); gp_tprs.append(tpr)

mean_fpr, gp_tpr_arr = interp_roc(gp_fprs, gp_tprs)
mean_tpr = gp_tpr_arr.mean(0); std_tpr = gp_tpr_arr.std(0)
gp_auc   = np.mean([auc(f, t) for f, t in zip(gp_fprs, gp_tprs)])

ax.plot(mean_fpr, mean_tpr, color=METHOD_COLORS["GP (ours)"], lw=2.2,
        label=f"{METHOD_LABELS['GP (ours)']}  (AUC={gp_auc:.3f})")
ax.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr,
                color=METHOD_COLORS["GP (ours)"], alpha=0.15)

# ── Baseline ROC ──
for bname, bdf in baseline_dfs.items():
    bpath = EVAL_DIR.parent / "baseline_results" / bname
    if not bpath.exists():
        continue
    b_fprs, b_tprs = [], []
    for _, row in bdf.iterrows():
        fpath = bpath / f"{row['fold_id']}.json"
        if not fpath.exists():
            continue
        with open(fpath) as f:
            br = json.load(f)
        novel = np.array(br["test_scores"])
        known = np.array(br.get("known_scores", []))
        if len(known) == 0:
            continue
        fpr, tpr, _ = fold_roc(novel, known)
        if fpr is not None:
            b_fprs.append(fpr); b_tprs.append(tpr)
    if b_fprs:
        _, b_tpr_arr = interp_roc(b_fprs, b_tprs)
        b_mean = b_tpr_arr.mean(0); b_std = b_tpr_arr.std(0)
        b_auc  = np.mean([auc(f, t) for f, t in zip(b_fprs, b_tprs)])
        c      = METHOD_COLORS[bname]
        ax.plot(mean_fpr, b_mean, color=c, lw=1.8,
                label=f"{METHOD_LABELS[bname]}  (AUC={b_auc:.3f})")
        ax.fill_between(mean_fpr, b_mean - b_std, b_mean + b_std, color=c, alpha=0.12)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Defect-only Labelling\n(macro-averaged, 72 folds, shading = ±1 SD)")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(FIG_DIR / "F3_roc_curves.png")
plt.show()
print("Saved F3_roc_curves.png")


## F4  [REPORT §6] — Statistical Comparison Table (Wilcoxon + Holm)

**Caption:** Two-sided Wilcoxon signed-rank test comparing GP AUROC against each
baseline across 72 matched folds, with Holm-Bonferroni correction.
* = significant at α = 0.05 after correction.


In [ ]:
tests = summary["wilcoxon_tests"]
tdf = pd.DataFrame(tests)[["method", "statistic", "p_value", "p_holm", "significant_holm"]]
tdf.columns = ["Baseline", "W-statistic", "p-value", "p (Holm)", "Sig. (Holm)"]
tdf["Sig. (Holm)"] = tdf["Sig. (Holm)"].map({True: "✓*", False: "✗"})
tdf["p-value"] = tdf["p-value"].apply(lambda x: f"{x:.4f}")
tdf["p (Holm)"] = tdf["p (Holm)"].apply(lambda x: f"{x:.4f}")

print(tdf.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, max(2, len(tdf) * 0.7 + 1)))
ax.axis("off")
tbl = ax.table(cellText=tdf.values, colLabels=tdf.columns,
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5)
tbl.scale(1.2, 1.8)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1565C0"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EEF2FF")

ax.set_title("Wilcoxon Signed-rank Test with Holm Correction", pad=18, fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "F4_wilcoxon_table.png", bbox_inches="tight")
plt.show()
print("Saved F4_wilcoxon_table.png")


## F5  [REPORT §8] — Held-out Severity vs AUROC

**Caption:** Each point is one LODTO fold. x-axis = mean severity of the held-out
novel type; y-axis = AUROC (defect-only).  If high-severity folds cluster at high
AUROC the GP may exploit intensity, not novelty; absence of correlation confirms
the signal is geometric (embedding space), not photometric.


In [ ]:
# parse test_severity (stored as list string or float)
def mean_sev(val):
    if isinstance(val, str):
        import ast
        try: return np.mean(ast.literal_eval(val))
        except: return np.nan
    return float(val)

fold_df["mean_test_sev"] = fold_df["test_severity"].apply(mean_sev)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    fold_df["mean_test_sev"], fold_df["auroc_do"],
    c=fold_df["auroc_do"], cmap="RdYlGn", vmin=0.3, vmax=1.0,
    s=55, alpha=0.75, edgecolors="k", linewidths=0.3,
)
plt.colorbar(sc, ax=ax, label="AUROC (defect-only)")
ax.axhline(0.5, ls="--", color="gray", lw=1)

# correlation annotation
valid = fold_df[["mean_test_sev","auroc_do"]].dropna()
r = np.corrcoef(valid["mean_test_sev"], valid["auroc_do"])[0,1]
ax.text(0.97, 0.05, f"r = {r:.2f}", transform=ax.transAxes,
        ha="right", fontsize=10, color="#333")

ax.set_xlabel("Mean defect severity of held-out type")
ax.set_ylabel("AUROC (defect-only)")
ax.set_title("Severity vs Novelty-detection AUROC (72 LODTO folds)")

plt.tight_layout()
plt.savefig(FIG_DIR / "F5_severity_vs_auroc.png")
plt.show()
print(f"Saved F5_severity_vs_auroc.png   (Pearson r = {r:.3f})")


## F6  [REPORT §4] — Kernel Selection Frequency

**Caption:** Frequency with which each of the six candidate kernels was selected
as optimal (highest log marginal likelihood) across all 72 LODTO folds.
Kernel selection is performed independently per fold; no single kernel
dominates, indicating severity surfaces vary in smoothness across categories.


In [ ]:
LABEL = {
    "rbf":     "RBF (SE)",
    "mat12":   "Matérn-½",
    "mat32":   "Matérn-³⁄₂",
    "mat52":   "Matérn-⁵⁄₂",
    "rq":      "Rational Q.",
    "lin_rbf": "Linear+RBF",
}
COLORS = ["#1565C0","#2E7D32","#F57F17","#AD1457","#6A1B9A","#00695C"]

if "kernel_name" in fold_df.columns:
    counts = fold_df["kernel_name"].value_counts()
    names  = [LABEL.get(n, n) for n in counts.index]
    clrs   = [COLORS[i % len(COLORS)] for i in range(len(counts))]
    total  = counts.sum()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(names, counts.values / total * 100, color=clrs, edgecolor="white")
    for bar, c in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                str(c), ha="center", fontsize=9)
    ax.set_ylabel("% of folds selected (by LML)")
    ax.set_title("Kernel Selection Frequency — 72 LODTO Folds")
    ax.set_ylim(0, counts.values.max() / total * 100 + 10)

    plt.tight_layout()
    plt.savefig(FIG_DIR / "F6_kernel_selection.png")
    plt.show()
    print("Saved F6_kernel_selection.png")
else:
    print("kernel_name column not found in fold_results.csv — re-run phase 3 with updated code.")


## A1  [APPENDIX] — Per-fold Detection Rate per Category


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
fold_df.boxplot(column="detection_rate", by="category", ax=ax,
                medianprops={"color": "tomato", "lw": 2},
                whiskerprops={"lw": 1.2}, capprops={"lw": 1.2},
                flierprops={"marker": ".", "markersize": 4})
ax.set_xticklabels([l.get_text() for l in ax.get_xticklabels()],
                   rotation=38, ha="right")
ax.set_title("Detection Rate per Category (cost-optimal threshold)")
ax.set_ylabel("Detection rate")
ax.axhline(0.5, ls="--", color="gray", lw=1)
plt.suptitle("")
plt.tight_layout()
plt.savefig(FIG_DIR / "A1_detection_rate_boxplot.png")
plt.show()
print("Saved A1_detection_rate_boxplot.png")


## A2  [APPENDIX] — AUROC vs PCA Dimension

**Caption:** Effect of PCA latent dimension d ∈ {8, 12, 16} on mean AUROC (defect-only).
Error bars = ±1 SD. This panel justifies the chosen value of d in the main results.


In [ ]:
dims, means, stds = [], [], []
for d in [8, 12, 16]:
    p = EVAL_DIR / f"gp_pca{d}.json"
    if not p.exists():
        continue
    with open(p) as f:
        dr = json.load(f)
    aucs = [r.get("auroc_do", np.nan) for r in dr]
    dims.append(d); means.append(np.nanmean(aucs)); stds.append(np.nanstd(aucs))

if len(dims) > 1:
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.errorbar(dims, means, yerr=stds, fmt="o-", color="#1565C0",
                capsize=5, lw=2, markersize=7)
    ax.axhline(0.5, ls="--", color="gray", lw=1)
    ax.set_xticks(dims)
    ax.set_xlabel("PCA latent dimension d")
    ax.set_ylabel("AUROC (defect-only)")
    ax.set_title("PCA Dimensionality vs AUROC")
    ax.set_ylim(0.3, 1.0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "A2_pca_dim_sweep.png")
    plt.show()
    print("Saved A2_pca_dim_sweep.png")
else:
    print("Only one PCA dimension run so far — run phases 3 for d=8 and d=12 to enable this plot.")


## [CODE ONLY] — Diagnostics (not cited in paper)

In [ ]:
if "lml" in fold_df.columns:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(fold_df["lml"].dropna(), bins=20, color="steelblue", edgecolor="white")
    ax.set_xlabel("Log Marginal Likelihood (best kernel + restart)")
    ax.set_title("GP LML Distribution across 72 Folds")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "DIAG_lml_distribution.png")
    plt.show()


In [ ]:
# Raw GP variance distributions: novel vs known
with open(EVAL_DIR / "gp_pca16.json") as f:
    gp_raw = json.load(f)

all_novel, all_known = [], []
for r in gp_raw:
    all_novel.extend(r["test_scores"])
    all_known.extend(r.get("known_scores", []))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(np.log1p(all_known), bins=60, alpha=0.65, color="#1565C0", label="Known defectives", density=True)
ax.hist(np.log1p(all_novel), bins=60, alpha=0.65, color="#B71C1C", label="Novel defectives", density=True)
ax.set_xlabel("log(1 + GP posterior variance)")
ax.set_ylabel("Density")
ax.set_title("GP Variance Distribution: Novel vs Known Defectives")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "DIAG_score_distributions.png")
plt.show()


## Summary

In [ ]:
print("=" * 65)
print("RESULTS NOTEBOOK COMPLETE")
print("=" * 65)

inventory = [
    ("[REPORT]  ", "M1_gp_uncertainty_surface.png"),
    ("[REPORT]  ", "F1_gp_vs_baselines_auroc.png"),
    ("[REPORT]  ", "F2_auroc_heatmap.png"),
    ("[REPORT]  ", "F3_roc_curves.png"),
    ("[REPORT]  ", "F4_wilcoxon_table.png"),
    ("[REPORT]  ", "F5_severity_vs_auroc.png"),
    ("[REPORT]  ", "F6_kernel_selection.png"),
    ("[APPENDIX]", "A1_detection_rate_boxplot.png"),
    ("[APPENDIX]", "A2_pca_dim_sweep.png"),
]
for tag, fname in inventory:
    exists = (FIG_DIR / fname).exists()
    print(f"  {tag} {fname:<45} {'✓' if exists else '✗ MISSING'}")
